# N082 · 拓扑排序与依赖图

**目标：** 判断是否存在合法依赖顺序并处理多解。

**先修：** N079, N080, N081。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 入度；Kahn；三色DFS；DAG；环；偏序与全序。

**配套讲解来源：** [同名逐章意见](../../comment/082_topological_sort_dag.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

生活中到处是"先 A 后 B"的依赖：先修课先上、先装依赖库再装主程序、先穿袜子再穿鞋。把每件事画成一个点、每个"u 必须在 v 前"画成一条有向边 u→v，问题就变成：**求一个做事顺序，让每条边的箭头都指向后面**——这就是拓扑排序。如果图里有环（A 等 B、B 等 A），合法顺序不存在，我们要能检测出来。

三个具体问题（同一套代码全解决）：

1. **求拓扑序：** n=4 个点、边 `[(0,1),(0,2),(1,3),(2,3)]`，一个合法输出是 `[0,1,2,3]`——0 在 1、2 前面，1、2 都在 3 前面，每条边都被尊重。注意 `[0,2,1,3]` 也合法：**拓扑序常常不唯一**，任何满足所有先后约束的顺序都算对。
2. **能否修完所有课（can_finish）：** n 门课、prerequisites 里每条 `[课程, 先修课]` 表示"想上这门课得先修那门"。n=2、`[[1,0]]` 返回 True（先修 0 再修 1 即可）；`[[1,0],[0,1]]` 返回 False（0 等 1、1 等 0，死锁，图有环）。
3. **给出修课顺序（find_order）：** 同上，但要返回一个具体顺序；有环时返回空列表 `[]`。n=4、无先修要求时，任何 0..3 的排列都行。

## 2. 基础知识：先读懂这些词

- **DAG（有向无环图）：** 边带方向且不存在环的图。只有 DAG 才有拓扑序——有环就意味着"A 要在 B 后、B 又要在 A 后"，自相矛盾。
- **拓扑排序：** 把所有点排成一列，使每条有向边 u→v 里 u 都排在 v 前面。它排的是"偏序关系"：没有约束的点之间随意。
- **偏序与全序：** 输入的约束只规定部分点对的先后（偏序），输出必须是一条完整序列（全序）。没有约束的两点谁先谁后都行，这就是"多解"的来源。
- **入度（indegree）：** 指向某点的边数。在依赖图里，入度 = "还有几个前置条件没满足"。入度 0 意味着万事俱备，现在就能做。
- **Kahn 算法（队列版拓扑排序）：** 反复做一件事——把所有入度 0 的点拿出来放进结果，然后删掉它们的出边（相当于这些条件"已完成"），删边后新变成入度 0 的点继续拿。拿完了所有点就成功；还有剩的就有环。
- **"删除出边"的懒惰实现：** 真去删边列表很麻烦，实际只把邻居的入度计数减 1，效果等价——入度减到 0 就等于"指向它的边都失效了"。
- **环的判据（剩点必有前驱）：** 如果算法停下时还有点没输出，这些点里每一个的入度都大于 0，也就是都还差前置条件。顺着"它等谁、谁又等谁"一路回溯，点有限、不重复，必然绕回出发点形成环。
- **三色 DFS（备选方法）：** “学习目标”部分 知识点里提到的另一种拓扑排序：DFS 完成时逆序记录，也能得到拓扑序，同时灰色点判环（N080 讲过三色）。本章实现用的是 Kahn 法。

## 3. 思路推导：从小例子开始

**Step 1：建图并统计入度。** 每条边 u→v：adj[u] 追加 v，indegree[v] 加 1。扫完所有边后，入度数组就是"每件事还差几个前置"的清单。

**Step 2：所有入度 0 的点入队。** 它们没有任何未满足的前置，是当前可以立即输出的候选。

**Step 3：出队、记录、削边。** 弹出一个点 u 放进 order；对 u 的每个邻居 v，入度减 1（u 完成了，v 少等一个）；减到 0 就入队（v 的前置全部到位）。

**Step 4：看总量下结论。** order 长度等于 n：全部输出，成功，order 就是拓扑序；长度不足 n：剩下的点困在环里，返回 None 表示"不存在合法顺序"。

**手算演示（topological_sort(4, [(0,1),(0,2),(1,3),(2,3)])，notebook 小实例）：**

初始：adj=`[[1,2],[3],[3],[]]`，indegree=`[0,1,1,2]`（3 号被 1 和 2 指着，要等两个）。

| 时刻 | 队列 | 弹出 | order | 入度变化 |
|---|---|---|---|---|
| 1 | [0] | 0 | [0] | 1 号 1→0 入队；2 号 1→0 入队 |
| 2 | [1,2] | 1 | [0,1] | 3 号 2→1 |
| 3 | [2] | 2 | [0,1,2] | 3 号 1→0 入队 |
| 4 | [3] | 3 | [0,1,2,3] | 无 |

order 长度 4 == n，返回 `[0,1,2,3]`。若边里再加一条 (3,0)，则 0 的入度变 1，初始队列为空，循环根本不启动，order=[] 长度 0 < 4，返回 None——环被"长度不足"暴露。

**手算演示（can_finish 的边方向反转）：** prerequisites 给的是 `[课程,先修课]`，比如 `[[1,0]]` 表示"1 这门课要先修 0"。依赖边应该是 `先修 → 课程` 即 0→1，所以构图时要做 `[(required,course) for course,required in prerequisites]` 的反转。忘反转的话边全反，判环结果在纯环检测下碰巧一样，但输出的 order 全部倒序，find_order 就错了——这是本章实现里最容易踩的坑，“正确性与复杂度”部分 特意强调。

## 4. 核心代码：topological_sort

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def topological_sort(n, edges):
    adj = [[] for _ in range(n)]
    indegree = [0] * n
    for u, v in edges:
        adj[u].append(v)
        indegree[v] += 1
    queue = deque((i for i in range(n) if indegree[i] == 0))
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
    return order if len(order) == n else None
```

### 逐段读懂代码

**第一段：topological_sort——Kahn 算法主体。**

```python
from collections import deque

def topological_sort(n, edges):
    adj = [[] for _ in range(n)]
    indegree = [0] * n
    for u, v in edges:
        adj[u].append(v)
        indegree[v] += 1
    queue = deque((i for i in range(n) if indegree[i] == 0))
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
    return order if len(order) == n else None
```

- `from collections import deque`（代码块首行）：队列用双端队列实现，`popleft()` 出队是 O(1)。用普通 list 的 `pop(0)` 出队是 O(n)，大图上会慢一个量级——和 N081 的选择同理。
- 前两行：建邻接表 + 数入度，一次循环同时完成（`adj[u].append(v); indegree[v]+=1` 用分号挤在一行，两件事：记边、记账）。
- `queue=deque(i for i in range(n) if indegree[i]==0)`：初始队列直接用生成器筛出所有零入度点。孤立点（没有边）入度自然是 0，会被排进序列——它们没有任何约束，出现在任何位置都合法。
- while 循环三步曲：弹出 u、追加进 order、把 u 的每个邻居入度减 1；哪个邻居减到 0，就立刻入队（它的前置全齐了）。
- 注意"删边"是**懒惰的**：代码里没有真的从 adj 里删掉任何东西，只是把邻居的入度计数减 1。入度减到 0 的语义就是"指向它的边全部作废"，两种说法效果一样，但记账便宜得多。
- `return order if len(order)==n else None`：三元表达式做最终裁决。输出了 n 个点 → 拓扑序有效；不足 n → 剩下的点互相等，存在环，返回 None 让调用方明确知道"无解"。

**第二段和第三段：can_finish 与 find_order——同一算法的两种包装。**

```python
def can_finish(n, prerequisites):
    return topological_sort(n, [(required, course) for course, required in prerequisites]) is not None

def find_order(n, prerequisites):
    order = topological_sort(n, [(required, course) for course, required in prerequisites])
    return [] if order is None else order
```

- 两个函数第一件事都是列表推导 `[(required,course) for course,required in prerequisites]`：把 LeetCode 风格的 `[课程,先修课]` 解包后重新打包成 `(先修,课程)` 边。`for course,required in prerequisites` 是序列解包，直接给两个变量起好名字，避免用 `p[0]`、`p[1]` 时搞反方向。
- `can_finish` 只关心有没有环：`is not None` 把"排出了完整序列"翻译成布尔值。
- `find_order` 关心顺序本身：有环按题意返回空列表 `[]`（注意和 None 区分开——对外契约是空列表，对内用 None 表示失败）。
- 两个函数一行核心逻辑，体现"一个问题（拓扑排序）多种问法（能否完成/给出顺序）"——算法写一次，包装随题意换。

## 5. 分段实现：按顺序运行

**本章接口：** `topological_sort(n, edges)`、`can_finish(n, prerequisites)`、`find_order(n, prerequisites)`

### 导入本章使用的库

In [1]:
from collections import deque

### topological_sort

In [2]:
def topological_sort(n, edges):
    adj = [[] for _ in range(n)]
    indegree = [0] * n
    for u, v in edges:
        adj[u].append(v)
        indegree[v] += 1
    queue = deque((i for i in range(n) if indegree[i] == 0))
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
    return order if len(order) == n else None

### can_finish

In [3]:
def can_finish(n, prerequisites):
    return topological_sort(n, [(required, course) for course, required in prerequisites]) is not None

### find_order

In [4]:
def find_order(n, prerequisites):
    order = topological_sort(n, [(required, course) for course, required in prerequisites])
    return [] if order is None else order

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1),(0,2),(1,3),(2,3)]; order=topological_sort(4,edges)
show_table(['位置','顶点','必须在它前面的顶点'],[(i,v,[u for u,w in edges if w==v]) for i,v in enumerate(order)])

位置,顶点,必须在它前面的顶点
0,0,[]
1,1,[0]
2,2,[0]
3,3,"[1, 2]"


## 7. 正确性、适用条件与复杂度

当前入度精确等于来自未输出顶点的前驱数。每次取零入度点不会违反任何边；无法继续且未结束则证明存在有向环。

**代价：** O(V+E)时间与空间。prerequisites使用[course,required]输入顺序，构图时需反转。

### 展开理解

**为什么每次取零入度点是安全的：** 入度 0 的含义是"没有任何还没完成的前置条件"。把它放进序列开头部分不会违反任何边——因为没有边要求别的点排在它前面。换句话说，任何合法拓扑序里，此刻的零入度点放最前都不出错。

**为什么入度计数准确反映"剩余约束"：** 我们维护的不变量是：任意时刻，点 v 的入度恰好等于"已输出且指向 v 的边之外还剩多少条指向它的边"。输出 u 时把 u 的所有出边对应的目标入度减 1，账目实时对上。所以"减到 0"和"所有前置已输出"永远说的是同一件事。

**为什么长度不足就是有环：** 算法停住意味着队列空了、没有点能再输出。剩下的每个点入度都大于 0，也就是各自还有未输出的前置。随便挑一个剩下的点，走向它没输出的前置，再走向前置的前置……剩下的点总共只有有限个，一路走下去必然会重访某个点——重访就意味着绕回来了，这条链上存在有向环。反过来，无环的 DAG 必然每次都至少有一个零入度点可取（否则同上论证出环，矛盾），所以一定能输出全部 n 个点。

**多解为什么不影响正确性：** 队列里同时有几个零入度点时，先弹谁取决于入队顺序，不同实现产生不同序列——但每个序列都满足"每条边 u→v 中 u 在 v 前"，都合法。这就是 “思路推导”部分 说"测试检查每条边顺序而不是强行匹配一份列表"的原因：测试只验证约束，不锁死唯一答案。

**复杂度（用具体数字感受）：** 时间 O(V+E)、空间 O(V+E)：每个点入队出队各一次，每条边只在"删边"时被碰一次。1 万门课、3 万条先修关系，约 4 万次基本操作，毫秒级。对比一个朴素错误做法——每次都全图扫一遍找零入度点——那是 O(V²+E)，1 万点就是上亿次操作，直接超时。

**备选方法：三色 DFS 也能排拓扑序。** “学习目标”部分 的知识点里还列了"三色 DFS"，这里补一句它和 Kahn 的关系（N080 详细讲过三色）。做法是：对每个白点做 DFS，点染灰进路径、邻居全探完后染黑**并把点压进一个结果栈**；全部做完后把栈倒过来就是拓扑序。为什么倒序正确：一个点被压栈时，它所有能到达的后继都已经先压进去了，所以它必然排在后继的前面——恰好是拓扑序要求的"u 在 v 前"。遇到灰点即判环，对应 Kahn 里"长度不足 n"。两种方法复杂度同为 O(V+E)，Kahn 迭代直观、天然给出队列序，三色版和 N080 的判环代码复用度高——本章实现选了 Kahn，但两条路你都要走得通。

**一个容易忽略的前提：** Kahn 的"入度"只统计**当前图**里指向 v 的边。代码里入度在建图时一次数好、之后只做减法，从不重算——这依赖"每个点至多被输出一次、它的出边只被削一次"这个流程纪律。如果你中途修改了 edges 或重复输出某点，账本就毁了，这也是为什么 while 循环里每个 u 严格出队一次。

## 8. 单元测试：每个用例在检查什么

```python
assert can_finish(2, [[1, 0]]) and (not can_finish(2, [[1, 0], [0, 1]]))
```
一正一反两个最小用例：单条依赖 0→1 无环可完成；0→1 加 1→0 组成二元环必须返回 False。这条同时测**边方向反转**是否正确——如果反转写反，两个结果都会颠倒。

```python
assert topological_sort(0, []) == [] and topological_sort(1, [(0, 0)]) is None
```
两个**边界值**：0 个点应该返回空序列而不是 None 或报错；自环 (0,0) 让 0 号的入度永远是 1（它自己等自己），队列空转、order 为空，必须返回 None。自环是"最小号的环"，专抓"忘了自环也算环"的实现。

```python
for n in range(1, 25):
    permutation = list(range(n))
    rng.shuffle(permutation)
    edges = [(permutation[i], permutation[j]) for i in range(n) for j in range(i + 1, n) if rng.randrange(5) == 0]
    order = topological_sort(n, edges)
    position = {x: i for i, x in enumerate(order)}
    assert len(order) == n and all((position[u] < position[v] for u, v in edges))
```
**随机对拍**（24 组）：先随机打乱生成一个排列，再按"排列中靠前的点指向靠后的点"随机抽一些边——这样构造的图按构造方式保证无环。验证两条：输出确实是 n 个点的排列；对每条边 u→v 检查 `position[u]<position[v]`（u 的位置在 v 前面）。注意它**没有**断言 order 等于某个固定列表——因为拓扑序不唯一，验证"所有约束被满足"才是对的测法（呼应第五节的多解讨论）。

```python
assert set(find_order(4, [])) == set(range(4))
```
零约束的正常值：4 门课没有先修要求，任何排列都合法，所以只检查结果**集合**恰好是 {0,1,2,3}（顺序不限）。这也再次体现"多解只验集合/约束"的测试哲学。

In [6]:
assert can_finish(2, [[1, 0]]) and (not can_finish(2, [[1, 0], [0, 1]]))

assert topological_sort(0, []) == [] and topological_sort(1, [(0, 0)]) is None

from random import Random

rng = Random(82)

for n in range(1, 25):
    permutation = list(range(n))
    rng.shuffle(permutation)
    edges = [(permutation[i], permutation[j]) for i in range(n) for j in range(i + 1, n) if rng.randrange(5) == 0]
    order = topological_sort(n, edges)
    position = {x: i for i, x in enumerate(order)}
    assert len(order) == n and all((position[u] < position[v] for u, v in edges))

assert set(find_order(4, [])) == set(range(4))

print('N082: 所有本章断言通过')

N082: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 返回任一合法顺序而不是指定顺序。

**练习 2：** 扩展输出每门课最早层次。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（返回任一合法顺序而不是指定顺序）：**

- 思路方向：本题官方答案也不唯一，你要做的是说服自己"验证比生成容易"。
- 提示：写一个独立函数 `is_valid_order(order, edges)`，逐条边检查 u 的下标小于 v 的下标；然后对若干输入断言 is_valid_order(find_order(...)) 为 True，而不是断言 find_order(...) == [某个你手算的列表]。
- 手算示例：边 [(0,1),(0,2),(1,3),(2,3)]，先列出所有合法序（[0,1,2,3] 和 [0,2,1,3]——想一想为什么只有这两个：0 必须第一，3 必须最后，中间 1、2 两种排法），再运行看返回的是哪一个。

**练习 2（输出每门课的最早层次）：**

- 思路方向："层次" = 最少要排在第几个学期，等于它所有先修课层次的最大值加 1。
- 提示：改造 Kahn 循环——队列元素带上 level，输出 u 时给邻居的"候选层次"取 max(邻居当前层次, level(u)+1)；入度减到 0 时邻居的层次就定稿了。
- 手算示例：边 [(0,1),(0,2),(1,3),(2,3)]，层次应为 0:0、1:1、2:1、3:2——3 号两门先修分属第 1 层，所以 3 号在第 2 层。
- 延伸思考：这个"逐层定稿"的过程和 N081 的 BFS 层次扩散有什么神似之处？提示：零入度队列的入队顺序本身就是按层次推进的。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from collections import deque

def topological_sort(n, edges):
    adj = [[] for _ in range(n)]
    indegree = [0] * n
    for u, v in edges:
        adj[u].append(v)
        indegree[v] += 1
    queue = deque((i for i in range(n) if indegree[i] == 0))
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
    return order if len(order) == n else None

def can_finish(n, prerequisites):
    return topological_sort(n, [(required, course) for course, required in prerequisites]) is not None

def find_order(n, prerequisites):
    order = topological_sort(n, [(required, course) for course, required in prerequisites])
    return [] if order is None else order

# 示例与回归测试
assert can_finish(2, [[1, 0]]) and (not can_finish(2, [[1, 0], [0, 1]]))

assert topological_sort(0, []) == [] and topological_sort(1, [(0, 0)]) is None

from random import Random

rng = Random(82)

for n in range(1, 25):
    permutation = list(range(n))
    rng.shuffle(permutation)
    edges = [(permutation[i], permutation[j]) for i in range(n) for j in range(i + 1, n) if rng.randrange(5) == 0]
    order = topological_sort(n, edges)
    position = {x: i for i, x in enumerate(order)}
    assert len(order) == n and all((position[u] < position[v] for u, v in edges))

assert set(find_order(4, [])) == set(range(4))

print('N082: 所有本章断言通过')

N082: 所有本章断言通过


## 11. 代表题与迁移

- **207. Course Schedule（canonical）：** 就是 can_finish，练"prerequisites 边方向反转 + Kahn 判环（输出不足 n 即有环）"。
- **210. Course Schedule II（canonical）：** 就是 find_order，额外要求返回具体顺序，练"拓扑序不唯一、按边约束验证结果"的思维。

**题面入口：** [207](https://leetcode.com/problems/course-schedule/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。